# PII controls with Presidio: detection, masking, and purpose-specific views

The synthetic text contains an email, an Indian mobile number, and an internal customer ID. We will not put the raw text into the evidence artifact.

In [ ]:
from hashlib import sha256
from pathlib import Path
import hmac
import importlib.util
import json
import re

import pandas as pd
from workshop_utils import redact_for_logs

raw_text = "Customer Asha Rao (CUST-48291) can be reached at asha.rao@example.test or +91 9876543210. Return window?"
print(raw_text)

## 1. Minimal offline fallback

This deterministic fallback is useful for unit tests and known identifiers. It is deliberately narrow and should not be presented as a complete PII detector.

In [ ]:
fallback_view = redact_for_logs(raw_text)
print(fallback_view)
assert "asha.rao@example.test" not in fallback_view
assert "9876543210" not in fallback_view

## 2. Presidio recognizers without an external NLP model

Pattern recognizers are a good fit for deterministic enterprise identifiers and locale-specific formats. A production deployment can combine them with Presidio's NLP recognizers and context.

In [ ]:
presidio_installed = importlib.util.find_spec("presidio_analyzer") is not None
print("Presidio installed:", presidio_installed)

presidio_result = None
entities = []
if presidio_installed:
    from presidio_analyzer import Pattern, PatternRecognizer
    from presidio_anonymizer import AnonymizerEngine
    from presidio_anonymizer.entities import OperatorConfig

    recognizers = [
        PatternRecognizer(
            supported_entity="EMAIL_ADDRESS",
            patterns=[Pattern("email", r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}", 0.85)],
        ),
        PatternRecognizer(
            supported_entity="IN_PHONE_NUMBER",
            patterns=[Pattern("india-mobile", r"(?<!\d)(?:\+?91[-\s]?)?[6-9]\d{9}(?!\d)", 0.85)],
        ),
        PatternRecognizer(
            supported_entity="CUSTOMER_ID",
            patterns=[Pattern("customer-id", r"\bCUST-\d{5}\b", 0.95)],
        ),
    ]
    analyzer_results = []
    for recognizer in recognizers:
        analyzer_results.extend(
            recognizer.analyze(
                text=raw_text,
                entities=[recognizer.supported_entities[0]],
                nlp_artifacts=None,
            )
        )
    analyzer_results.sort(key=lambda r: r.start)
    entities = [
        {"entity_type": r.entity_type, "start": r.start, "end": r.end, "score": r.score}
        for r in analyzer_results
    ]
    operators = {
        "EMAIL_ADDRESS": OperatorConfig("replace", {"new_value": "<EMAIL>"}),
        "IN_PHONE_NUMBER": OperatorConfig("replace", {"new_value": "<PHONE>"}),
        "CUSTOMER_ID": OperatorConfig("replace", {"new_value": "<CUSTOMER_ID>"}),
    }
    presidio_result = AnonymizerEngine().anonymize(
        text=raw_text, analyzer_results=analyzer_results, operators=operators
    ).text
    print(presidio_result)
    display(pd.DataFrame(entities))
else:
    print("Install pinned dependencies: pip install -r requirements.txt")

## 3. Create different views for different purposes

A stable pseudonym supports correlation without copying the original customer ID into general telemetry. In production, keep the HMAC key in a secrets manager and version/rotate it deliberately.

In [ ]:
def pseudonymize(value: str, key: bytes) -> str:
    return "psn_" + hmac.new(key, value.encode(), sha256).hexdigest()[:16]

customer_id = re.search(r"CUST-\d{5}", raw_text).group(0)
views = {
    "model_input": presidio_result or fallback_view.replace("CUST-48291", "<CUSTOMER_ID>"),
    "general_log": redact_for_logs(raw_text).replace("CUST-48291", "<CUSTOMER_ID>"),
    "fraud_linkage": pseudonymize(customer_id, b"synthetic-workshop-key-do-not-use"),
}
print(json.dumps(views, indent=2))

In [ ]:
# Contract: evidence and ordinary destinations contain no raw identifiers.
for destination in ["model_input", "general_log"]:
    value = views[destination]
    assert "asha.rao@example.test" not in value
    assert "9876543210" not in value
    assert "CUST-48291" not in value
assert views["fraud_linkage"].startswith("psn_")

# Do not export raw_text. Store entity types/scores and transformed views only.
evidence = {
    "raw_text_sha256": sha256(raw_text.encode()).hexdigest(),
    "detected_entities": entities,
    "views": views,
    "controls": {
        "pre_egress_transformation": True,
        "raw_value_in_evidence": False,
        "pseudonym_key_storage": "demo only; use managed secrets in production",
    },
}
out = Path("_evidence/05_pii_views.json")
out.parent.mkdir(exist_ok=True)
out.write_text(json.dumps(evidence, indent=2), encoding="utf-8")
print("PASS: raw direct identifiers are absent from exported views")
print("Wrote", out.resolve())

## What to test next

Build a labeled test set containing local names, addresses, account formats, multilingual text, OCR errors, and adversarial separators. Track false negatives separately from over-redaction because both can cause harm: leakage on one side, unusable service or disparate failure on the other.